In [ ]:
include("../src/TensorDecomposition.jl")
using LinearAlgebra, LinearSolve

In [40]:
n = 5
r = 16

D, Drev = TensorDecomposition.makeDicts(n, 4);
basis_inds = collect(1:r)
basis, basisD = TensorDecomposition.basisFn(basis_inds, Drev);

vars = TensorDecomposition.varTups(basis, n, 4)
eqs1, eqs2 = TensorDecomposition.linEqTups(basisD, n, 4);

In [41]:
length(vars)

90

In [47]:
length(eqs1)

85

In [42]:
length(eqs1)+length(eqs2)

106

The number of equations exceeds the number of variables.  Note that in our code we already preprocess the equations to remove duplicates and equations that are differences of two other equations.  Of course, some linearly dependent subsets of equations may remain of which we do not know the relationship.  Note that $\mathbf{A}^1$ is *not* tall.

In [45]:
Z = randn(n+1, r) / sqrt(n)
T = TensorDecomposition.rankedTensor(ones(r), Z, 4; type=eltype(Z));

Tcat = TensorDecomposition.catMat(T, 2)
H0 = Tcat[basis_inds, basis_inds]

Z_ = Z ./ permutedims(Z[1, :]);
permutedims(Z_)

16×6 Matrix{Float64}:
 1.0   0.848602    -0.683925   -1.63231      0.928905   1.45255
 1.0   8.20084     -3.19473     6.51795     -9.55605   -3.03039
 1.0   0.389885    -0.74869    -1.38231     -0.169059   0.10175
 1.0  -1.66103     -0.486849   -0.900408    -0.284591   2.58604
 1.0  -2.95149     -3.60732    -0.756261     8.2184     5.65027
 1.0  -0.908582    -2.61126    -0.108769    -1.46946    0.60364
 1.0  -0.443245    -1.08951     0.0256962    1.39637   -0.111045
 1.0   7.48995     -0.91765    -0.444341   -10.6773     1.23442
 1.0   0.933205     0.017146    1.38868     -0.813254  -2.46704
 1.0   1.16295      1.28582     2.12246      2.53194   -0.555131
 1.0   5.78406     -1.37101    13.0774      -0.161016  11.848
 1.0  -3.72578     -9.67876   -11.4989      -4.93092   -8.97157
 1.0   2.18819     -3.50363    -0.497438     1.11009   -0.802375
 1.0  -0.226991     7.42533     2.84482     -1.34203   -0.693336
 1.0   0.0060224   -0.535927   -0.381207     0.196344   0.964677
 1.0   7.79941 

In [46]:
A, b = TensorDecomposition.linearSystem(T, H0, basis_inds, basisD, D, vars, eqs1, eqs2; type=eltype(T));

A_ = Matrix(copy(A))
foreach(TensorDecomposition.normalize!, eachcol(A_));
foreach(TensorDecomposition.normalize!, eachrow(A_));
svdvals(A_)

90-element Vector{Float64}:
 2.6453468487369682
 2.581774968486478
 2.279652695284169
 2.201860909252934
 2.1621811884302793
 2.1409390408784965
 2.0653696752179496
 2.049566121977765
 2.0236070281988123
 1.983254704419943
 1.9695891237983667
 1.9310452728122725
 1.8938124297572998
 ⋮
 0.045319736772672974
 0.04250124167004656
 0.0395098526333658
 0.027196981913464875
 0.023783707241339515
 0.007943343516930523
 0.00570581500859747
 0.0039350691664766894
 0.0016276308173581884
 0.001107886900667057
 0.0007978927903843297
 2.1066097260258513e-16

Although $\mathbf{A}$ is tall it is not full column rank numerically.